In [40]:
import pandas as pd
import numpy as np
from pathlib import Path


## Phase 1. Generate 100K unknown data
---
Store `unknown_car.data.csv` in $data$ folder

In [41]:
def generate_unknown_data(n_rows: int = 100_000, random_state: int = 42) -> None:
    """
    Introduce 'unknown' values randomly in the DataFrame.

    Parameters:
    - n_rows: Number of rows to generate.
    - random_state: Seed for reproducibility.

    Returns:
    - None, 
    creates a csv file 'unknown_car_data.csv' in /data folder
    """
    # Set random seed for reproducibility
    np.random.seed(random_state)

    # Define categories based on UCI Car Dataset
    data_config = {
        'price': ['vhigh', 'high', 'med', 'low'],
        'maintenance': ['vhigh', 'high', 'med', 'low'],
        'doors': ['2', '3', '4', '5more'],
        'seats': ['2', '4', 'more'],
        'storage': ['small', 'med', 'big'],
        'safety': ['low', 'med', 'high'],
        'shouldBuy': ['unacc', 'acc', 'vgood', 'good']
    }

    # Define imbalanced weights for 'shouldBuy' (Target)
    # Original UCI distribution is roughly: unacc (~70%), acc (~22%), good (~4%), vgood (~4%)
    target_classes = ['unacc', 'acc', 'good', 'vgood']
    target_weights = [0.70, 0.22, 0.04, 0.04]


    # Create base synthetic data with random features
    data = {col: np.random.choice(vals, size=n_rows) for col, vals in data_config.items()}

    # Generate imbalanced target labels
    data['shouldBuy'] = np.random.choice(target_classes, size=n_rows, p=target_weights)

    df_imbalanced = pd.DataFrame(data)

    # Function to inject various types of missingness (The "Chaos" from before)
    def inject_chaos(df, p=0.05):
        n = len(df)
        for col in df.columns:
            indices = np.random.choice(n, size=int(n * p), replace=False)
            idx_splits = np.array_split(indices, 5)
            
            df.loc[idx_splits[0], col] = np.nan
            df.loc[idx_splits[1], col] = None
            df.loc[idx_splits[2], col] = ""
            df.loc[idx_splits[3], col] = "null"
            df.loc[idx_splits[4], col] = "NaN"
        return df

    # Inject 5% chaos to maintain the real-world difficulty
    df_imbalanced_dirty = inject_chaos(df_imbalanced, p=0.05)

    # Setup path and save
    results_dir = Path.cwd().parent / "data"
    results_dir.mkdir(parents=True, exist_ok=True)
    file_path = results_dir / "unknown_car_data.csv"

    df_imbalanced_dirty.to_csv(file_path, index=False)

    # Verification
    print(f"Generated {n_rows} rows of imbalanced unknown data.")
    print(f"File saved to: {file_path}")
    print("\nTarget Class Distribution (Should match UCI proportions):")
    print(df_imbalanced_dirty['shouldBuy'].value_counts(normalize=True))

    return df_imbalanced_dirty

# Phase 2: Test the models
--- 
We created models in classification.ipynb; we must test them here

In [55]:
# Load Data
df_unknown = pd.read_csv(Path.cwd().parent / "data" / "unknown_car_data.csv")
display(df_unknown.sample(10))

,price,maintenance,doors,seats,storage,safety,shouldBuy
28453,high,med,2,more,big,NaN,unacc
96799,med,high,5more,more,med,high,unacc
36327,high,med,3,4,med,med,unacc
59936,high,low,4,2,med,high,unacc
15010,high,vhigh,2,more,big,med,acc
63078,med,high,5more,4,NaN,med,good
18930,vhigh,med,NaN,2,med,high,unacc
64591,vhigh,low,4,2,small,med,acc
3141,vhigh,high,3,2,med,low,acc
5410,high,vhigh,3,more,small,med,unacc


In [61]:
df_unknown.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   price        95000 non-null  object
 1   maintenance  95000 non-null  object
 2   doors        95000 non-null  object
 3   seats        95000 non-null  object
 4   storage      95000 non-null  object
 5   safety       95000 non-null  object
 6   shouldBuy    95000 non-null  object
dtypes: object(7)
memory usage: 5.3+ MB


In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import classification_report

# 1. Setup Pathing
results_dir = Path.cwd().parent / "results"
data_path = results_dir / "unknown_car_data_dirty.csv"

# 2. Load the messy 100k data
df_unknown = pd.read_csv(data_path)

# 3. UNIFY NULLS: Crucial step to turn strings into technical nulls
null_flavors = ["", "null", "NaN", "NULL", "none", "None"]
df_cleaned = df_unknown.replace(null_flavors, np.nan)

# Separate features and actual labels for validation
X_stress = df_cleaned.drop(columns=['shouldBuy'])
y_stress_actual = df_cleaned['shouldBuy']